# Census–EOD data inspection and harmonization

This checkpoint loads the required 2023 EOD and 2020 Census Expanded Questionnaire tables, selects and renames the relevant fields, and reports their schemas and missing values. It does not construct the target, harmonize predictors, aggregate persons, or save processed outputs.

In [1]:
from importlib.metadata import version

import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from eodgdl import load_eod
from mxcensus import load_extended_personas, load_extended_viviendas

semilla = 2026
np.random.seed(semilla)
pd.set_option("display.max_rows", 30)
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 120)

versiones_fuentes = {"eodgdl": version("eodgdl"), "mxcensus": version("mxcensus"), "pandas": version("pandas")}
display(Markdown("**Source package versions:** " + ", ".join(f"`{nombre} {numero}`" for nombre, numero in versiones_fuentes.items())))

**Source package versions:** `eodgdl 0.1.0`, `mxcensus 0.3.1`, `pandas 3.0.5`

## Load source tables

The EOD loader returns four linked tables; only the dwelling and person tables are retained. The Census loaders retrieve the Jalisco dwelling and person tables from the Expanded Questionnaire.

In [2]:
eod_viviendas_original, eod_personas_original, eod_viajes_descartado, eod_tramos_descartado = load_eod()
del eod_viajes_descartado, eod_tramos_descartado

censo_viviendas_original = load_extended_viviendas(state=14)
censo_personas_original = load_extended_personas(state=14)

municipios_interes = [39, 44, 51, 70, 97, 98, 101, 120, 124]
censo_viviendas_original = censo_viviendas_original.loc[censo_viviendas_original["MUN"].isin(municipios_interes)]
censo_personas_original = censo_personas_original.loc[censo_personas_original["MUN"].isin(municipios_interes)]

filas_fuente = pd.DataFrame({"tabla": ["eod_viviendas", "eod_personas", "censo_viviendas", "censo_personas"], "filas": [len(eod_viviendas_original), len(eod_personas_original), len(censo_viviendas_original), len(censo_personas_original)]})
display(Markdown("**Rows loaded for the study area**"))
display(filas_fuente)

**Rows loaded for the study area**

,tabla,filas
0,eod_viviendas,17901
1,eod_personas,58061
2,censo_viviendas,39238
3,censo_personas,141954


## Select and rename relevant fields

EOD field names are already in Spanish `snake_case`. Census source names are converted to descriptive Spanish `snake_case` without changing their values or meanings.

In [3]:
columnas_eod_viviendas = ["ponderador", "municipio", "ageb", "centralidad", "personas_en_vivienda", "tenencia_vivienda", "ingreso_mensual_hogar", "n_autos_camionetas", "n_motos", "n_bicicletas", "donde_estacionan", "tiene_internet"]
columnas_eod_personas = ["parentesco", "sexo_nacimiento", "edad", "estado_civil", "escolaridad", "trabajo_semana_pasada", "ocupacion"]
columnas_censo_viviendas = ["ENT", "MUN", "LOC50K", "COBERTURA", "ESTRATO", "UPM", "FACTOR", "CLAVIVP", "NUMPERS", "TENENCIA", "DEUDA", "AUTOPROP", "MOTOCICLETA", "BICICLETA", "INTERNET", "INGTRHOG", "JEFE_SEXO", "JEFE_EDAD", "TAMLOC"]
columnas_censo_personas = ["ENT", "MUN", "LOC50K", "COBERTURA", "ESTRATO", "UPM", "FACTOR", "CLAVIVP", "NUMPER", "SEXO", "EDAD", "PARENTESCO", "NIVACAD", "ESCOLARI", "ESCOACUM", "SITUA_CONYUGAL", "CONACT", "OCUPACION_C", "SITTRA", "INGTRMEN"]

renombres_censo_viviendas = {"ENT": "entidad", "MUN": "municipio", "LOC50K": "localidad_50_mil", "COBERTURA": "cobertura", "ESTRATO": "estrato", "UPM": "upm", "FACTOR": "factor_expansion", "CLAVIVP": "clavivp", "NUMPERS": "numero_personas", "TENENCIA": "tenencia", "DEUDA": "deuda", "AUTOPROP": "automovil_camioneta", "MOTOCICLETA": "motocicleta", "BICICLETA": "bicicleta", "INTERNET": "internet", "INGTRHOG": "ingreso_trabajo_hogar", "JEFE_SEXO": "sexo_jefatura", "JEFE_EDAD": "edad_jefatura", "TAMLOC": "tamano_localidad"}
renombres_censo_personas = {"ENT": "entidad", "MUN": "municipio", "LOC50K": "localidad_50_mil", "COBERTURA": "cobertura", "ESTRATO": "estrato", "UPM": "upm", "FACTOR": "factor_expansion", "CLAVIVP": "clavivp", "NUMPER": "numero_persona", "SEXO": "sexo", "EDAD": "edad", "PARENTESCO": "parentesco", "NIVACAD": "nivel_academico", "ESCOLARI": "grado_escolar", "ESCOACUM": "escolaridad_acumulada", "SITUA_CONYUGAL": "situacion_conyugal", "CONACT": "condicion_actividad", "OCUPACION_C": "clave_ocupacion", "SITTRA": "posicion_trabajo", "INGTRMEN": "ingreso_trabajo_mensual"}

eod_viviendas = eod_viviendas_original[columnas_eod_viviendas].copy()
eod_personas = eod_personas_original[columnas_eod_personas].copy()
censo_viviendas = censo_viviendas_original[columnas_censo_viviendas].rename(columns=renombres_censo_viviendas).rename_axis("id_vivienda_censo").copy()
censo_personas = censo_personas_original[columnas_censo_personas].rename(columns=renombres_censo_personas).rename_axis(index={"ID_VIV": "id_vivienda_censo", "ID_PERSONA": "id_persona_censo"}).copy()

## Selected schemas and missing values

Missingness is reported directly from each selected table. No source codes are recoded at this checkpoint.

In [4]:
def mostrar_resumen(nombre_tabla, tabla):
    resumen_columnas = pd.DataFrame({"tipo_dato": tabla.dtypes.astype(str), "nulos": tabla.isna().sum(), "porcentaje_nulos": (tabla.isna().mean() * 100).round(2)})
    indices = ", ".join(nombre or "sin_nombre" for nombre in tabla.index.names)
    display(Markdown(f"### `{nombre_tabla}`\n\nRows: **{len(tabla):,}** · Columns: **{tabla.shape[1]}** · Index: `{indices}`"))
    display(resumen_columnas)

mostrar_resumen("eod_viviendas", eod_viviendas)
mostrar_resumen("eod_personas", eod_personas)
mostrar_resumen("censo_viviendas", censo_viviendas)
mostrar_resumen("censo_personas", censo_personas)

### `eod_viviendas`

Rows: **17,901** · Columns: **12** · Index: `folio_vivienda`

,tipo_dato,nulos,porcentaje_nulos
ponderador,int64,0,0.00
municipio,category,0,0.00
ageb,str,0,0.00
centralidad,category,0,0.00
personas_en_vivienda,category,0,0.00
tenencia_vivienda,category,0,0.00
ingreso_mensual_hogar,category,0,0.00
n_autos_camionetas,category,0,0.00
n_motos,category,0,0.00
n_bicicletas,category,0,0.00


### `eod_personas`

Rows: **58,061** · Columns: **7** · Index: `folio_vivienda, folio_habitante`

,tipo_dato,nulos,porcentaje_nulos
parentesco,category,377,0.65
sexo_nacimiento,category,0,0.00
edad,int64,0,0.00
estado_civil,category,341,0.59
escolaridad,category,10442,17.98
trabajo_semana_pasada,category,4370,7.53
ocupacion,category,4110,7.08


### `censo_viviendas`

Rows: **39,238** · Columns: **19** · Index: `id_vivienda_censo`

,tipo_dato,nulos,porcentaje_nulos
entidad,int64,0,0.00
municipio,int64,0,0.00
localidad_50_mil,int64,0,0.00
cobertura,category,0,0.00
estrato,str,0,0.00
upm,int64,0,0.00
factor_expansion,int64,0,0.00
clavivp,category,0,0.00
numero_personas,int64,0,0.00
tenencia,category,0,0.00


### `censo_personas`

Rows: **141,954** · Columns: **20** · Index: `id_vivienda_censo, id_persona_censo`

,tipo_dato,nulos,porcentaje_nulos
entidad,int64,0,0.00
municipio,int64,0,0.00
localidad_50_mil,int64,0,0.00
cobertura,category,0,0.00
estrato,str,0,0.00
upm,int64,0,0.00
factor_expansion,int64,0,0.00
clavivp,category,0,0.00
numero_persona,int64,0,0.00
sexo,category,0,0.00


## Construct the Census target

The original labeled `clavivp` field is preserved unchanged. A separate code column provides explicit traceability to the official values, and `dwelling_type` groups codes 1–9 into four physical dwelling categories. Code 99 remains an unknown target and is not assigned to a physical category.

In [ ]:
codigos_clavivp = {"Casa única en el terreno": 1, "Casa que comparte terreno con otra(s)": 2, "Casa dúplex": 3, "Departamento en edificio": 4, "Vivienda en vecindad o cuartería": 5, "Vivienda en cuarto de azotea de un edificio": 6, "Local no construido para habitación": 7, "Vivienda móvil": 8, "Refugio": 9, "No especificado de vivienda particular": 99}
censo_viviendas["clavivp_codigo"] = censo_viviendas["clavivp"].map(codigos_clavivp).astype("Int64")

distribucion_clavivp = censo_viviendas.groupby(["clavivp_codigo", "clavivp"], observed=True)["factor_expansion"].agg(registros="size", frecuencia_ponderada="sum").reset_index().sort_values("clavivp_codigo")
distribucion_clavivp["porcentaje_ponderado"] = (distribucion_clavivp["frecuencia_ponderada"] / distribucion_clavivp["frecuencia_ponderada"].sum() * 100).round(2)
display(Markdown("**Observed `CLAVIVP` distribution before grouping**"))
display(distribucion_clavivp)

**Observed `CLAVIVP` distribution before grouping**

,clavivp_codigo,clavivp,registros,frecuencia_ponderada,porcentaje_ponderado
0,1,Casa única en el terreno,30604,1090030,74.25
1,2,Casa que comparte terreno con otra(s),3534,152061,10.36
2,3,Casa dúplex,1744,46014,3.13
3,4,Departamento en edificio,3044,166056,11.31
4,5,Vivienda en vecindad o cuartería,216,10184,0.69
5,7,Local no construido para habitación,38,1526,0.10
6,8,Vivienda móvil,1,4,0.00
7,9,Refugio,13,402,0.03
8,99,No especificado de vivienda particular,44,1855,0.13


In [6]:
# We group the dwelling types into four categories: "casa_unica", "casa_compartida_duplex", "departamento", and "otra_vivienda". The mapping is as follows: 
agrupacion_dwelling_type = {1: "casa_unica", 2: "casa_compartida_duplex", 3: "casa_compartida_duplex", 4: "departamento", 5: "otra_vivienda", 6: "otra_vivienda", 7: "otra_vivienda", 8: "otra_vivienda", 9: "otra_vivienda"}
# We avoid the unusual dwelling types by mapping them to "otra_vivienda".
categorias_dwelling_type = ["casa_unica", "casa_compartida_duplex", "departamento", "otra_vivienda"]
censo_viviendas["dwelling_type"] = censo_viviendas["clavivp_codigo"].map(agrupacion_dwelling_type).astype(pd.CategoricalDtype(categorias_dwelling_type))

grupo_reporte = censo_viviendas["dwelling_type"].astype("string").fillna("objetivo_desconocido")
distribucion_dwelling_type = censo_viviendas.assign(grupo_reporte=grupo_reporte).groupby("grupo_reporte")["factor_expansion"].agg(registros="size", frecuencia_ponderada="sum").reindex(categorias_dwelling_type + ["objetivo_desconocido"]).reset_index().rename(columns={"grupo_reporte": "dwelling_type"})
distribucion_dwelling_type["porcentaje_ponderado"] = (distribucion_dwelling_type["frecuencia_ponderada"] / distribucion_dwelling_type["frecuencia_ponderada"].sum() * 100).round(2)
display(Markdown("**Weighted target distribution after grouping**"))
display(distribucion_dwelling_type)

**Weighted target distribution after grouping**

,dwelling_type,registros,frecuencia_ponderada,porcentaje_ponderado
0,casa_unica,30604,1090030,74.25
1,casa_compartida_duplex,5278,198075,13.49
2,departamento,3044,166056,11.31
3,otra_vivienda,268,12116,0.83
4,objetivo_desconocido,44,1855,0.13


## Save validated Stage 1 tables

The selected dwelling tables are saved with their source-specific identifiers. The Census output includes the original `clavivp`, its traceable numeric code, and the validated four-category target. No person attributes or remaining predictors are harmonized at this stage.

In [7]:
from pathlib import Path

ruta_outputs = Path("..") / "outputs"
ruta_censo = ruta_outputs / "censo_viviendas_armonizado.csv"
ruta_eod = ruta_outputs / "eod_viviendas_armonizado.csv"
censo_viviendas.to_csv(ruta_censo, index=True)
eod_viviendas.to_csv(ruta_eod, index=True)

censo_exportado = pd.read_csv(ruta_censo)
eod_exportado = pd.read_csv(ruta_eod)
validacion_salidas = pd.DataFrame({"archivo": [ruta_censo.name, ruta_eod.name], "identificador": ["id_vivienda_censo", "folio_vivienda"], "filas": [len(censo_exportado), len(eod_exportado)], "columnas": [censo_exportado.shape[1], eod_exportado.shape[1]], "una_fila_por_vivienda": [not censo_exportado["id_vivienda_censo"].duplicated().any(), not eod_exportado["folio_vivienda"].duplicated().any()]})
columnas_salidas = pd.DataFrame({"archivo": [ruta_censo.name, ruta_eod.name], "columnas_finales": [", ".join(censo_exportado.columns), ", ".join(eod_exportado.columns)]})
display(Markdown("**Final Stage 1 output validation**"))
display(validacion_salidas)
display(Markdown("**Final column names**"))
display(columnas_salidas)

**Final Stage 1 output validation**

,archivo,identificador,filas,columnas,una_fila_por_vivienda
0,censo_viviendas_armonizado.csv,id_vivienda_censo,39238,22,True
1,eod_viviendas_armonizado.csv,folio_vivienda,17901,13,True


**Final column names**

,archivo,columnas_finales
0,censo_viviendas_armonizado.csv,"id_vivienda_censo, entidad, municipio, localid..."
1,eod_viviendas_armonizado.csv,"folio_vivienda, ponderador, municipio, ageb, c..."
